This notebook compiles and organizes the effect size data. 

Last updated on January 28, 2025, by Ruimin Gao.

In [1]:
import os
import pandas as pd
import pathlib
import numpy as np
cwd = pathlib.Path('/Users/rgao76/Documents/DiffTasks/effect_estimation_2025')

In [2]:
effects = [f'langloc_DiffTasks_{i}' for i in range(1, 7)]
localizers = ['MD_spatialFIN', 'DMN_spatialFIN'] + [
    'language_' + effect for effect in effects
]
effects = effects + ['spatialFIN']

data = []
for localizer in localizers:
    for effect in effects:
        pth = cwd / "Data" / f"{localizer}_{effect}.csv"
        if not pth.exists():
            print(f"File not found: {pth}")
            continue
        df = pd.read_csv(pth)
        df['System'] = 'MD' if 'MD' in localizer else 'DMN' if 'DMN' in localizer else 'Language'
        df['Localizer'] = f"V{localizer[-1]}" if 'language' in localizer else df['System']
        df['Task'] = f"V{effect[-1]}" if effect != 'spatialFIN' else ''
        df['Subject'] = df['Subject'].str.split('_').str[0]

        if 'DMN' in localizer:
            df['ROI'] = df['ROI'].map({
                1: 'LH_FrontalMed',
                2: 'LH_PostCing',
                3: 'LH_TPJ',
                4: 'LH_MidCing',
                5: 'LH_STGorInsula',
                6: 'LH_AntTemp',
                7: 'RH_FrontalMed',
                8: 'RH_PostCing',
                9: 'RH_TPJ',
                10: 'RH_MidCing',
                11: 'RH_STGorInsula',
                12: 'RH_AntTemp'
            })
            
        df['Hemisphere'] = df['ROI'].str.split('_').str[0]

        data.append(df)

data = pd.concat(data)

In [3]:
data['Type'] = data.apply(
    lambda row: 'TwoSentences' if row['Task'] in ['V4','V5','V6'] and row['Effect'] == 'S' else 'OneSentence',
    axis=1
)
data['SentType'] = data.apply(
    lambda row: 'TwoSentences' if row['Task'] in ['V4','V5','V6'] else 'OneSentence',
    axis=1
)
data['Version'] = data['Task']
data['Task'] = data.apply(
        lambda row: 'ButtonPress' if row['Version'] == 'V1' or (row['Version'] == 'V6' and row['Effect'] == 'S')
        else 'MemoryAll' if row['Version'] == 'V2' or (row['Version'] in ['V4','V5','V6'] and row['Effect'] == 'N')
        else 'MemoryLast' if row['Version'] == 'V3'
        else 'ComprehensionQ' if row['Version'] == 'V4'
        else 'Sentiment',
        axis=1
    )
data['SentTask'] = data.apply(
        lambda row: 'ButtonPress' if row['Version'] == 'V1' or row['Version'] == 'V6'
        else 'MemoryAll' if row['Version'] == 'V2'
        else 'MemoryLast' if row['Version'] == 'V3'
        else 'ComprehensionQ' if row['Version'] == 'V4'
        else 'Sentiment',
        axis=1
    )

In [4]:
data.to_csv(cwd / 'Data' / 'all_data.csv', index=False)